# Finding Potential Structural Relatives by Sequence Similarity using proteusPy
Eric G. Suchanek, PhD 10/9/24

Working under the assumption that similar sequence -> similar structure I generated a query on the lowest energy Disulfide Bond in the RCSB database (2q7q) to return PDB IDs for structures with high sequence similarity. I then use some of the proteusPy functions to find structures with similar disulfide bonds.

In [ ]:
#
import pandas as pd
import pyvista as pv

from proteusPy import DisulfideList, Load_PDB_SS

# pyvista setup for notebooks
pv.set_jupyter_backend("trame")

# set_plot_theme("dark")
LIGHT = "auto"

### Load the RCSB Disulfide Database
We load the database and get its properties as follows:

In [15]:
PDB_SS = Load_PDB_SS(verbose=True)

[04/26/25 00:39:23] INFO     proteusPy: INFO 2025-04-26 00:39:23,384 -                      ]8;id=12078;file:///Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages/proteusPy/DisulfideLoader.py\DisulfideLoader.py]8;;\:]8;id=992017;file:///Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages/proteusPy/DisulfideLoader.py#1175\1175]8;;\
                             proteusPy.DisulfideLoader.Load_PDB_SS - Done reading                                  
                             disulfides from:                                                                      
                             /Users/egs/miniforge3/envs/ppydev/lib/python3.12/site-packages                        
                             /proteusPy/data/PDB_SS_ALL_LOADER.pkl...                                              


    🌟 RCSB Disulfide Database Summary 🌟
       🕒 Constructed: 2025-04-25 21:27:45 🕒
PDB IDs Present:               36968
Disulfides Loaded:             175277
Average Resolution:            2.19 Å
Lowest Energy Disulfide:       2q7q_75D_140D
Highest Energy Disulfide:      6vxk_801B_806B
Cα Distance Cutoff:            -1.00 Å
Sγ Distance Cutoff:            -1.00 Å
Percentile Cutoff:             -1.00 %
     ⚡ proteusPy Version: 0.99.35.dev1 ⚡



In [17]:
best_ss = PDB_SS["2q7q_75D_140D"]
best_ss.pprint()
best_ss.display(style="sb", light=LIGHT)

<Disulfide 2q7q_75D_140D, Source: 2q7q, Resolution: 1.0 Å 
Χ1-Χ5: -59.36°, -59.28°, -83.66°, -59.82° -59.91°, 0.17°, 0.49 kcal/mol 
Cα Distance: 5.50 Å 
Sγ Distance: 2.04 Å 
Torsion length: 145.62 deg>


Widget(value='<iframe src="http://localhost:59762/index.html?ui=P_0x1759c2360_4&reconnect=auto" class="pyvista…

I generated a query on: https://www.ebi.ac.uk/pdbe/entry/pdb/2q7q to return PDB IDs for structures with high sequence similarity to 2q7q - the protein with the lowest energy disulfide bond in the RCSB database. This yielded a ```.csv``` file, which we will import below:

In [18]:
ss_df = pd.read_csv("../data/2q7q_seqsim.csv")
ss_df.head(5)

,pdb_id,organism_scientific_name,tax_id,organism_synonyms,rank,genus,superkingdom,journal,journal_volume,journal_first_page,...,molecule_name,all_molecule_name,modified_residue_flag,molecule_type,mutation_type,entry_uniprot_accession,uniprot_id,molecule_synonym,gene_name,entity_id
0,2q7q,Paracoccus denitrificans,266,"Parde,Paracoccus Denitrificans,Micrococcus Den...","species,genus,family,order,class,phylum,superk...",Paracoccus,Bacteria,J. Mol. Biol.,276.0,NaN,...,Methylamine dehydrogenase heavy chain,NaN,N,Protein,Conflict,"P29894,P22619",DHMH_PARDE,"Methylamine dehydrogenase (amicyanin),Methylam...",mauB,1
1,2bbk,Paracoccus denitrificans,266,"Parde,Paracoccus Denitrificans,Micrococcus Den...","species,genus,family,order,class,phylum,superk...",Paracoccus,Bacteria,J. Mol. Biol.,276.0,NaN,...,Methylamine dehydrogenase light chain,NaN,Y,Protein,NaN,"P29894,P22619",DHML_PARDE,"Methylamine dehydrogenase (amicyanin),MADH,Met...",mauA,2
2,2agy,Alcaligenes faecalis,511,"Achromobacter Sp. Atcc8750,Alcaligenes Sp. Bp1...","species,genus,family,order,class,phylum,superk...",Alcaligenes,Bacteria,Science,312.0,NaN,...,Aralkylamine dehydrogenase light chain,NaN,Y,Protein,NaN,"P84887,P84888",AAUA_ALCFA,"Aromatic amine dehydrogenase,AADH,Aralkylamine...",aauA,1
3,2agy,Alcaligenes faecalis,511,"Achromobacter Sp. Atcc8750,Alcaligenes Sp. Bp1...","species,genus,family,order,class,phylum,superk...",Alcaligenes,Bacteria,Science,312.0,NaN,...,Aralkylamine dehydrogenase heavy chain,NaN,N,Protein,NaN,"P84887,P84888",AAUB_ALCFA,"Aromatic amine dehydrogenase,Aralkylamine dehy...",aauB,2
4,2ah1,Alcaligenes faecalis,511,"Achromobacter Sp. Atcc8750,Alcaligenes Sp. Bp1...","species,genus,family,order,class,phylum,superk...",Alcaligenes,Bacteria,Science,312.0,NaN,...,Aralkylamine dehydrogenase light chain,NaN,Y,Protein,NaN,"P84888,P84887",AAUA_ALCFA,"Aromatic amine dehydrogenase,AADH,Aralkylamine...",aauA,1


All of the nearest sequence neighbors are sadly, bacterial. Let's extract the unique ids next.

In [ ]:
relative_list = ss_df["pdb_id"].unique()
relative_list

We now need to convert the list of PDB IDs into real disulfides from the database. We do this with the ``DisulfideLoader.build_ss_from_idlist()`` function. Next we print out some relevant statistics.


In [ ]:
relatives = DisulfideList([], "relatives")
relatives = PDB_SS.build_ss_from_idlist(relative_list)

print(
    f"There are: {relatives.length} related structures.\nAverage Energy: {relatives.average_energy:.2f} kcal/mol\nAverage Ca distance: {relatives.average_distance:.2f} Å"
)
print(
    f"Average resolution: {relatives.average_resolution:.2f} Å \nAverage torsion distance: {relatives.average_torsion_distance:.2f}°"
)

Now let's look at the lowest and highest energy structures in this list of relatives.

In [ ]:
ssmin, ssmax = relatives.minmax_energy
duolist = DisulfideList([ssmin, ssmax], "mM")
# duolist.display(style='sb', light=LIGHT)

In [ ]:
duolist.display_overlay(light=LIGHT)

The two Disulfides vary considerably in their overall geometry!

We can find disulfides that are conformationally related by using the ``DisulfideList.nearest_neighbors()`` function with a dihedral angle cutoff. This cutoff is measure of angular similarity across all five sidechain dihedral angles and is the Euclidean distance between the two sets of dihedral angles.

In [ ]:
close_neighbors = relatives.nearest_neighbors(5.0, ssmin.dihedrals)
close_neighbors.length

In [ ]:
close_neighbors.display_overlay(light=LIGHT)

So now we have the 18 close neighbors of the lowest energy structure.

In [ ]:
ssTotList = PDB_SS.SSList
global_neighbors = ssTotList.nearest_neighbors(5.0, ssmin.dihedrals)
global_neighbors.length

In [ ]:
global_neighbors.display_overlay(light=LIGHT)